In [63]:
import glob
import pandas as pd
import os
from pathlib import Path
import glob
import shutil
import re
import unicodedata
import hashlib

In [64]:
def normalize_zh(text):
    return normalize(text, enc_ZH = True)

def normalize(text, enc_ZH = False):
    if pd.isnull(text):
        return ""
    # Lowercase
    text = text.lower()
    # Remove accents (e.g., é → e)
    if enc_ZH:
        #text = unicodedata.normalize('NFKD', text)
        text = hashlib.sha1(text.encode("utf-8")).hexdigest()[:8]
    else:
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
    # Remove punctuation
    text = re.sub(r"[^\w\s]", "", text)
    # Collapse multiple spaces
    text = re.sub(r"\s+", " ", text)
    # Strip leading/trailing spaces
    return text.strip().replace(' ', '_')

In [65]:
ROOT = '/Users/tomasandrade/Documents/BSC/ICHOIR/datasets/GTSinger'

In [66]:
lang = 'ZH'

In [67]:
output_folder = f'{ROOT}/GTSinger_{lang}_flat'

In [68]:
os.makedirs(f'{output_folder}/wav')
os.makedirs(f'{output_folder}/TextGrid')

In [ ]:
all_files = glob.glob(f'{ROOT}/{lang}/*/**/***/****/*****')
data = [file.split(f'{ROOT}/{lang}/')[1].split('/') for file in all_files]

df = pd.DataFrame(data = data, columns=['singer', 'style', 'song', 'group', 'file'])
df['input_path'] = all_files


# prepare output paths
# HERE CHOOSE ZH and non-ZH
df['song'] = df['song'].apply(normalize_zh)
df['new_song'] = df['style'] + '_' +  df['song'] + '_' + df['group']
df['out_path'] = df['singer'] + '-' + df['new_song'] + '-' + df['file'].str.replace('_TextGrid', '.TextGrid')

In [70]:
df['style'].unique()

array(['Vibrato', 'Pharyngeal', 'Breathy', 'Glissando',
       'Mixed_Voice_and_Falsetto'], dtype=object)

In [71]:
SELECT_STYLES = ['Vibrato']
EXCLUDE_GROUPS = ['Paired_Speech_Group', 'Control_Group']

df_songs = df[~df['group'].isin(EXCLUDE_GROUPS)]

df_songs = df_songs[df_songs['style'].isin(SELECT_STYLES)]

df_wav = df_songs[df_songs['file'].str.contains('wav')].sort_values('input_path')
df_wav['out_path'] = f'{ROOT}/GTSinger_{lang}_flat/wav/' + df_wav['out_path']

df_TextGrid = df_songs[df_songs['file'].str.contains('TextGrid')].sort_values('input_path')
df_TextGrid['out_path'] = f'{ROOT}/GTSinger_{lang}_flat/TextGrid/' + df_TextGrid['out_path']

In [72]:
df_wav

,singer,style,song,group,file,input_path,new_song,out_path
1059,ZH-Alto-1,Vibrato,0d2d2bfc,Vibrato_Group,0000.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_0d2d2bfc_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
1057,ZH-Alto-1,Vibrato,0d2d2bfc,Vibrato_Group,0001.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_0d2d2bfc_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
1046,ZH-Alto-1,Vibrato,0d2d2bfc,Vibrato_Group,0002.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_0d2d2bfc_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
1052,ZH-Alto-1,Vibrato,0d2d2bfc,Vibrato_Group,0003.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_0d2d2bfc_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
1030,ZH-Alto-1,Vibrato,0d2d2bfc,Vibrato_Group,0004.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_0d2d2bfc_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
...,...,...,...,...,...,...,...,...
19338,ZH-Tenor-1,Vibrato,1c25d6e2,Vibrato_Group,0016.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_1c25d6e2_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
19342,ZH-Tenor-1,Vibrato,1c25d6e2,Vibrato_Group,0017.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_1c25d6e2_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
19390,ZH-Tenor-1,Vibrato,1c25d6e2,Vibrato_Group,0018.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_1c25d6e2_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...
19385,ZH-Tenor-1,Vibrato,1c25d6e2,Vibrato_Group,0019.wav,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...,Vibrato_1c25d6e2_Vibrato_Group,/Users/tomasandrade/Documents/BSC/ICHOIR/datas...


In [73]:
df_wav.shape, df_TextGrid.shape

((512, 8), (512, 8))

In [74]:
for input_path, out_path in zip(df_wav['input_path'], df_wav['out_path']):
    shutil.copy2(input_path, out_path)

for input_path, out_path in zip(df_TextGrid['input_path'], df_TextGrid['out_path']):
    shutil.copy2(input_path, out_path)

In [75]:
lang

'ZH'

# Checks

In [76]:
def check_matching_files(lang):
    tg_root = f'{ROOT}/GTSinger_{lang}_flat/TextGrid'
    wav_root = f'{ROOT}/GTSinger_{lang}_flat/wav'

    tg_files = sorted(glob.glob(f'{tg_root}/*'))
    wav_files = sorted(glob.glob(f'{wav_root}/*'))

    tg_stems = [file.split('/')[-1].split('.TextGrid')[0] for file in tg_files]
    wav_stems = [file.split('/')[-1].split('.wav')[0] for file in wav_files]

    df_tg = pd.DataFrame(data = tg_stems)
    df_wav = pd.DataFrame(data = wav_stems)

    n_wav = len(df_wav)
    n_tg = len(df_tg)

    matches = (df_tg == df_wav).sum().values[0]

    return n_wav, n_tg, matches

In [77]:
check_matching_files('ZH')

(512, 512, 512)

In [78]:
from textgrids import TextGrid

In [79]:
def get_all_tg_files(lang):
    tg_root = f'{ROOT}/GTSinger_{lang}_flat/TextGrid'
    tg_files = sorted(glob.glob(f'{tg_root}/*'))

    return tg_files

def get_total_length(lang):
    files = get_all_tg_files(lang)
    secs = sum([get_tg_length(f) for f in files])
    return secs/60.

def get_tg_length(file):
    tg = TextGrid()
    tg.read(file)

    return tg['global'].xmax

In [80]:
get_total_length('ZH')

69.60601666666665